# BoneNano step 0b — scan positions by cross-correlation

DanMAX records CA2's motor positions in `scan_NN_positions.csv`, but the scan
80 NFP set has no such file, and a motor reading is in millimetres with an
unknown axis, sign and scale anyway. So the positions are measured from the
data, by registering each frame against the first.

Appends `/exchange/shifts` to `data_nfp.h5`, which is the last thing
`step0.py` needs.

Shifts are **(row, col) in pixels at this bin**, built exactly as the
phascade `bone_nano_holotomo` NFP notebook builds them:

1. divide by `pref` to strip the probe envelope, **then** by the frame mean
   to strip the static CA1 pattern -- without the second division the
   correlation locks onto CA1, which does not move;
2. `phase_corr` between CONSECUTIVE frames;
3. integrate from a **mid-scan anchor**, giving `pos[anchor] - pos[k]`,
   which is the engine's sign;
4. centre with `centered_full`.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('../../src'))
import numpy as np, cupy as cp, h5py
import matplotlib.pyplot as plt

from holotomocupy.config import parse_args_coded_step0
CFG = parse_args_coded_step0('config_step0.conf')   # paths from the config
H5 = CFG.in_file

with h5py.File(H5, 'r') as f:
    pdata = np.asarray(f['/exchange/pdata'], 'float32')
    pref  = np.asarray(f['/exchange/pref'],  'float32')
NPOS, nz, n = pdata.shape
print(f'{NPOS} positions, {nz} x {n}')

## Positions: chained registration from a mid-scan anchor

In [ ]:
def phase_corr(a, b):
    # Integer-pixel displacement of b relative to a.
    # Matches phascade.preprocess.phase_corr exactly.
    A, B = cp.fft.fft2(cp.asarray(a)), cp.fft.fft2(cp.asarray(b))
    cc = cp.fft.fftshift(cp.fft.ifft2(A * cp.conj(B)).real)
    i = cp.unravel_index(cp.argmax(cc), cc.shape)
    return np.array([int(i[0]) - cc.shape[0] // 2,
                     int(i[1]) - cc.shape[1] // 2], 'float32')

def integrate_shifts_from_anchor(rel, anchor):
    # out[k] = pos[anchor] - pos[k], from consecutive relative shifts.
    # Port of phascade.preprocess.integrate_shifts_from_anchor. The anchor is
    # mid-scan, not frame 0: consecutive frames correlate far better than
    # distant ones, so chaining from the middle keeps the longest path short.
    rel = np.asarray(rel)
    out = np.zeros_like(rel)
    for k in range(anchor):
        out[k] = np.sum(rel[k:anchor], axis=0)
    out[anchor] = 0
    for k in range(anchor + 1, len(rel)):
        out[k] = np.sum(-rel[anchor:k], axis=0)
    return out

def centered_full(po):
    # phascade.pipeline.coarse2fine.centered_full
    return po - (po.max(0) + po.min(0)) / 2

# TWO divisions, both needed -- this is the step that decides whether the
# registration sees CA2 at all:
#   / pref        strips the probe envelope (per pixel)
#   / rd.mean(0)  strips the STATIC CA1 pattern, which does not move and would
#                 otherwise dominate the correlation peak and pull every
#                 measured shift toward zero
rd = pdata / (pref[None] + 1e-6)
rd = rd / (rd.mean(0)[None] + 1e-6)

step = np.zeros((NPOS, 2), 'float32')
for k in range(NPOS - 1):
    step[k] = phase_corr(rd[k], rd[k + 1])

ipos = NPOS // 2                       # mid-scan anchor
shifts = integrate_shifts_from_anchor(step, ipos)
shifts = centered_full(shifts)

print(f'anchor={ipos}  span(px) row {np.ptp(shifts[:,0]):.1f} '
      f'col {np.ptp(shifts[:,1]):.1f}  max|shift| {np.abs(shifts).max():.1f}')

fig, ax = plt.subplots(1, 2, figsize=(11, 4.4))
ax[0].plot(shifts[:, 0], label='row'); ax[0].plot(shifts[:, 1], label='col')
ax[0].set_xlabel('position index'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(shifts[:, 1], shifts[:, 0], 'o-', ms=3, alpha=.7)
ax[1].plot(shifts[ipos, 1], shifts[ipos, 0], 'rx', ms=12, label='anchor')
ax[1].set_xlabel('col [px]'); ax[1].set_ylabel('row [px]')
ax[1].set_aspect('equal'); ax[1].grid(alpha=.3); ax[1].legend()
plt.tight_layout(); plt.show()

## Diagnostic: does a direct registration agree with the chained one?

Chaining through neighbours and registering straight against the anchor
are independent routes to the same number. They agree only if the
correlation is actually locking onto CA2.

In [ ]:
# Independent route: register every frame DIRECTLY against the anchor instead
# of chaining through its neighbours. phase_corr(rd[ipos], rd[k]) is the
# displacement of k relative to the anchor, and the chained result is
# pos[anchor] - pos[k], so the two differ by a sign.
direct = -np.stack([phase_corr(rd[ipos], rd[k]) for k in range(NPOS)])
direct = centered_full(direct)

err = np.abs(direct - shifts)
print(f'chained vs direct: mean |d| {err.mean():.3f} px, max {err.max():.3f} px')
print('Expect ~1 px. Tens of px means the registration is locking onto '
      'something other than CA2 -- check that the /rd.mean(0) division above '
      'really removed the static CA1 pattern.')

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for d, lab in enumerate(('row', 'col')):
    ax[d].plot(shifts[:, d], label='chained (used)')
    ax[d].plot(direct[:, d], '--', label='direct from anchor')
    ax[d].set_title(lab); ax[d].legend(); ax[d].grid(alpha=.3)
plt.tight_layout(); plt.show()

## Diagnostic: is the scan big enough to need the plane we configured?

`npsiz`/`npsi` must clear `nz`/`n` plus twice the largest shift plus a margin,
or `step0.py`'s FOV filter silently throws positions away. This prints what the
config should say.

In [ ]:
mx = np.abs(shifts).max(0)
need = np.array([nz, n]) + 2 * np.ceil(mx) + 2 * 16
print(f'max |shift|  rows {mx[0]:.0f}  cols {mx[1]:.0f}')
print(f'minimum npsiz x npsi = {need[0]:.0f} x {need[1]:.0f}')
print(f'rounded up to a power of two: '
      f'{1 << int(np.ceil(np.log2(need[0])))} x {1 << int(np.ceil(np.log2(need[1])))}')
print('-> put these in config_step0.conf as npsiz / npsi')

## Append to `data_nfp.h5`

In [ ]:
with h5py.File(H5, 'a') as f:
    if '/exchange/shifts' in f:
        del f['/exchange/shifts']
    f.create_dataset('/exchange/shifts', data=shifts.astype('float32'))
print('wrote /exchange/shifts', shifts.shape)
print('Next:  mpirun -np <ngpus> python step0.py config_step0.conf')